In [0]:
import json
import uuid
from datetime import date

class LandingWriter:
    def __init__(self, dbutils, base_path):
        self.dbutils = dbutils
        self.base_path = base_path

    def write(self, lote, collection, chave="_id"):
        if not lote:
            return None

        hoje = date.today().isoformat()
        path = f"{self.base_path}/{collection}/_ingestion_date={hoje}"
        self.dbutils.fs.mkdirs(path)

        linhas = []
        for doc in lote:
            registro = {
                "_source_id": str(doc.get(chave)),
                "_body": json.dumps(doc, default=str),
            }
            linhas.append(json.dumps(registro))

        file_path = f"{path}/{collection}_{uuid.uuid4().hex}.json"
        self.dbutils.fs.put(file_path, "\n".join(linhas), overwrite=True)
        return file_path